# How Much Can You Trust Your Forecasts?

A point forecast is only half the story. Stakeholders need to know **how confident**
you are in a prediction before they can act on it. This notebook walks through the
full uncertainty-quantification pipeline that `polars-ts` provides:

| Stage | Tools |
|---|---|
| **Generate intervals** | `QuantileRegressor`, `conformal_interval`, `EnbPI` |
| **Score** | `crps` (Continuous Ranked Probability Score) |
| **Diagnose** | `calibration_table`, `pit_histogram`, `reliability_diagram` |
| **Fix** | `bias_detect`, `bias_correct` |

We use the **M4 Hourly** dataset (3 series) throughout. By the end you will be
able to produce, evaluate, and improve prediction intervals for any time-series
forecasting workflow.

**References**
- Gneiting & Raftery (2007), *Strictly Proper Scoring Rules, Prediction, and Estimation*
- Xu & Xie (2021), *Conformal Prediction Interval for Dynamic Time-Series* (EnbPI)
- Kuleshov et al. (2018), *Accurate Uncertainties for Deep Learning Using Calibrated Regression*

## 1 — Imports & Data Loading

We pull three hourly series from the M4 competition hosted on S3 and split each
into **train** (all but last 48 steps) and **test** (last 48 steps).

In [ ]:
try:
    import hvplot.polars  # noqa
except ImportError:
    pass  # hvplot is optional for visualization
import polars as pl
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge

from polars_ts import (
    EnbPI,
    QuantileRegressor,
    bias_correct,
    bias_detect,
    calibration_table,
    conformal_interval,
    crps,
    holt_winters_forecast,
    mae,
    pit_histogram,
    reliability_diagram,
)

In [ ]:
HORIZON = 48
SERIES_IDS = ["H1", "H2", "H3"]

df = (
    pl.scan_parquet("https://datasets-nixtla.s3.amazonaws.com/m4-hourly.parquet")
    .filter(pl.col("unique_id").is_in(SERIES_IDS))
    .collect()
)

# Train / test split: last HORIZON rows per series go to test
train = df.group_by("unique_id", maintain_order=True).map_groups(lambda g: g.head(len(g) - HORIZON))
test = df.group_by("unique_id", maintain_order=True).map_groups(lambda g: g.tail(HORIZON))

print(f"Train: {train.shape}, Test: {test.shape}")
train.head()

## 2 — Baseline Point Forecasts (Holt-Winters)

Before quantifying uncertainty we need a **point forecast** to anchor our analysis.
Holt-Winters with a 24-hour seasonal period is a reasonable starting point for hourly data.

In [ ]:
hw_fcst = holt_winters_forecast(train, h=HORIZON, season_length=24)

# Join actuals for later evaluation
baseline = test.join(hw_fcst, on=["unique_id", "ds"], how="left")

print("MAE per series:")
mae(baseline, actual_col="y", predicted_col="y_hat")

## 3 — Quantile Regression

`QuantileRegressor` wraps any sklearn-compatible regressor that supports quantile
loss. We use `GradientBoostingRegressor(loss="quantile")` to produce five quantile
forecasts: 10%, 25%, 50% (median), 75%, and 90%.

In [ ]:
QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]

qr = QuantileRegressor(
    estimator_factory=lambda q: GradientBoostingRegressor(
        loss="quantile", alpha=q, n_estimators=100, max_depth=4, random_state=42
    ),
    quantiles=QUANTILES,
    lags=[1, 2, 3, 24],
)
qr.fit(train)
qr_preds = qr.predict(train, h=HORIZON)

# Merge with actuals
qr_eval = test.join(qr_preds, on=["unique_id", "ds"], how="left")
qr_eval.head(10)

In [ ]:
# Visualize prediction intervals for H1
h1 = qr_eval.filter(pl.col("unique_id") == "H1")

area_90 = h1.hvplot.area(x="ds", y="q_0.1", y2="q_0.9", alpha=0.2, color="steelblue", label="90% interval")
area_50 = h1.hvplot.area(x="ds", y="q_0.25", y2="q_0.75", alpha=0.35, color="steelblue", label="50% interval")
median_line = h1.hvplot.line(x="ds", y="q_0.5", color="navy", label="Median")
actual_line = h1.hvplot.scatter(x="ds", y="y", color="red", size=12, label="Actual")

(area_90 * area_50 * median_line * actual_line).opts(
    title="Quantile Regression — H1", ylabel="Value", width=800, height=350
)

## 4 — Conformal Prediction Intervals

Conformal prediction takes the **residuals from a calibration set** and uses their
empirical distribution to wrap any point forecast with a coverage guarantee.
No distributional assumptions required.

We split the training set further: the last 48 points become the **calibration set**.

In [ ]:
# Calibration set = last 48 rows of train per series
cal = train.group_by("unique_id", maintain_order=True).map_groups(lambda g: g.tail(HORIZON))
train_proper = train.group_by("unique_id", maintain_order=True).map_groups(lambda g: g.head(len(g) - HORIZON))

# Forecast on the calibration period
cal_fcst = holt_winters_forecast(train_proper, h=HORIZON, season_length=24)
cal_merged = cal.join(cal_fcst, on=["unique_id", "ds"], how="left")

# Compute absolute residuals
cal_residuals = cal_merged.with_columns((pl.col("y") - pl.col("y_hat")).abs().alias("residual"))

# Build conformal intervals around the test-set point forecasts
conf_preds = conformal_interval(cal_residuals, hw_fcst, coverage=0.9)
conf_eval = test.join(conf_preds, on=["unique_id", "ds"], how="left")

# Plot H1
h1_conf = conf_eval.filter(pl.col("unique_id") == "H1")

area_conf = h1_conf.hvplot.area(
    x="ds",
    y="y_hat_lower",
    y2="y_hat_upper",
    alpha=0.25,
    color="coral",
    label="90% conformal",
)
line_conf = h1_conf.hvplot.line(x="ds", y="y_hat", color="darkred", label="Point forecast")
scatter_conf = h1_conf.hvplot.scatter(x="ds", y="y", color="black", size=12, label="Actual")

(area_conf * line_conf * scatter_conf).opts(
    title="Conformal Prediction Intervals — H1 (90% coverage)",
    ylabel="Value",
    width=800,
    height=350,
)

## 5 — EnbPI: Ensemble Batch Prediction Intervals

**EnbPI** (Xu & Xie, 2021) is an online conformal method that adaptively updates
its residual distribution as new observations arrive. It naturally handles
non-stationarity and distribution shift.

In [ ]:
enbpi = EnbPI(
    estimator_factory=lambda: Ridge(alpha=1.0),
    quantiles=[0.1, 0.5, 0.9],
    lags=[1, 2, 3, 24],
)
enbpi.fit(train)
enbpi_preds = enbpi.predict(train, h=HORIZON)

enbpi_eval = test.join(enbpi_preds, on=["unique_id", "ds"], how="left")

h1_enb = enbpi_eval.filter(pl.col("unique_id") == "H1")
(
    h1_enb.hvplot.area(x="ds", y="q_0.1", y2="q_0.9", alpha=0.25, color="green", label="80% EnbPI")
    * h1_enb.hvplot.line(x="ds", y="q_0.5", color="darkgreen", label="Median")
    * h1_enb.hvplot.scatter(x="ds", y="y", color="black", size=12, label="Actual")
).opts(title="EnbPI — H1", ylabel="Value", width=800, height=350)

## 6 — Scoring with CRPS

The **Continuous Ranked Probability Score** (CRPS) is the gold-standard metric for
probabilistic forecasts. It generalises MAE to full predictive distributions and
is expressed in the same units as the target variable — lower is better.

In [ ]:
q_cols = [f"q_{q}" for q in QUANTILES]

crps_qr = crps(qr_eval, actual_col="y", quantile_cols=q_cols, quantiles=QUANTILES)
print("CRPS — Quantile Regression:")
crps_qr

## 7 — Calibration Analysis

A well-calibrated model's 90% interval should contain **90% of observations**.
Three diagnostic views help us check:

- **Calibration table**: expected vs. observed coverage at each quantile level.
- **PIT histogram**: Probability Integral Transform — should be uniform if calibrated.
- **Reliability diagram**: scatter of expected vs. observed coverage.

In [ ]:
# Calibration table
cal_tbl = calibration_table(qr_eval, actual_col="y")
print("Calibration Table:")
cal_tbl

In [ ]:
# PIT histogram
pit = pit_histogram(qr_eval, actual_col="y", n_bins=10)

pit.hvplot.bar(
    x="bin_lower",
    y="density",
    color="teal",
    alpha=0.7,
    width=600,
    height=300,
    title="PIT Histogram (should be roughly uniform)",
    xlabel="PIT bin",
    ylabel="Density",
)

In [ ]:
import holoviews as hv

# Reliability diagram
rel = reliability_diagram(qr_eval, actual_col="y")

diagonal = hv.Curve([(0, 0), (1, 1)], kdims="expected", vdims="observed").opts(
    color="gray", line_dash="dashed", line_width=1
)

points = rel.hvplot.scatter(
    x="expected",
    y="observed",
    color="navy",
    size=60,
    width=500,
    height=400,
    title="Reliability Diagram",
    xlim=(0, 1),
    ylim=(0, 1),
)

diagonal * points

## 8 — Bias Detection and Correction

Even a model with good interval coverage can have a **systematic bias** in its
point forecast. `bias_detect` summarises the direction and magnitude of bias,
and `bias_correct` applies an adjustment.

In [ ]:
# Detect bias in the Holt-Winters baseline
bias_info = bias_detect(baseline, actual_col="y", predicted_col="y_hat")
print("Bias diagnostics:")
bias_info

In [ ]:
# Correct the bias using the mean-error method
corrected = bias_correct(baseline, actual_col="y", predicted_col="y_hat", method="mean")

print("MAE before correction:")
print(mae(baseline, actual_col="y", predicted_col="y_hat"))

print("\nMAE after correction:")
print(mae(corrected, actual_col="y", predicted_col="y_hat"))

corrected.head()

## 9 — Summary & Next Steps

| Technique | Strengths | When to use |
|---|---|---|
| **Quantile Regression** | Native quantile estimates; flexible | You have enough data to train ML models |
| **Conformal Prediction** | Distribution-free; finite-sample guarantee | Wrap any existing point-forecast model |
| **EnbPI** | Online updates; handles non-stationarity | Streaming / production systems |

**Key takeaways**:

1. Always evaluate intervals with **CRPS**, not just point-forecast MAE.
2. Use **PIT histograms** and **reliability diagrams** to diagnose mis-calibration.
3. Check for **systematic bias** before deploying — a small mean-error correction
   can materially improve forecast value.

**Further reading**:

- Gneiting, T. & Raftery, A. (2007). *Strictly Proper Scoring Rules, Prediction, and Estimation*. JASA.
- Xu, C. & Xie, Y. (2021). *Conformal Prediction Interval for Dynamic Time-Series*. ICML.
- Kuleshov, V. et al. (2018). *Accurate Uncertainties for Deep Learning Using Calibrated Regression*. ICML.
- Jansen, M. (2024). *Modern Time Series Forecasting with Python*, 2nd Edition. Packt.